# GRPO 2 回目（LoRA、4B、Colab A100）

1 回目（`notebooks/colab_grpo_4b.ipynb`）からの変更:
- 学習セット `data/perception_test/train_grpo2.jsonl`（2,335 問）。count は「数字だけで答える」形式（type=numeric）で、正解との差 1 に 0.5 の部分点
- dev `data/perception_test/dev_grpo2.jsonl`（405 問、count 139・temporal 101）。学習セットと同じ分布、動画は重ならない
- 動的サンプリング: 全回答が同じ報酬の問題は飛ばして別の問題を引き、1 ステップに 4 問ぶんの学習信号を確保する（上限 12 問）

前提: Drive の `MyDrive/movie_llm/` に `pt_frames_f32_s640.zip`（Perception Test のフレーム）と `frames_f32_s640.zip`（Japanese Video-QA のフレーム）がある。

流れ: 学習前の dev → GRPO（チェックポイントは Drive に 20 ステップごと）→ 学習後の dev → ベンチマークの 4択・yes_no を学習後の LoRA で予測。
まず `--max-steps 20` で 1 ステップの時間を測り、問題なければ 300 に伸ばす。途中で切れても同じセルの再実行で最新のチェックポイントから続く。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers>=5.0" "accelerate>=1.0" "peft>=0.15" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "python-dotenv" "openai"
!pip uninstall -y -q torchao  # Colab 既定の古い torchao (0.10) を peft が拒否するので外す
!pip install -q -e . --no-deps

## 2. Drive とフレーム（Perception Test とベンチマークの両方）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
OUT = f'{DRIVE}/grpo2_4b'
import os
!mkdir -p /content/pt_frames /content/frames {OUT}
!unzip -q -o {DRIVE}/pt_frames_f32_s640.zip -d /content/pt_frames
!unzip -q -o {DRIVE}/frames_f32_s640.zip -d /content/frames
# 予測スクリプトは動画ファイルの有無で対象を決めるので、空のダミーを置く（フレームはキャッシュから読まれる）
for frames, videos in (('/content/pt_frames', '/content/pt_videos'), ('/content/frames', '/content/videos')):
    os.makedirs(videos, exist_ok=True)
    for vid in os.listdir(frames):
        open(f'{videos}/{vid}.mp4', 'a').close()
    print(frames, len(os.listdir(frames)), '本')

## 3. 学習前の dev（貪欲法、405 問。約 15 分）

In [ ]:
%cd /content/movie_llm
!python scripts/pt_difficulty.py --items data/perception_test/dev_grpo2.jsonl --out {OUT}/dev_before.jsonl \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --greedy --video-dir /content/pt_videos --frames-dir /content/pt_frames

## 4. GRPO 学習（まず 20 ステップで時間を測る → 300 に伸ばす）

In [ ]:
%cd /content/movie_llm
!python scripts/train_grpo.py --items data/perception_test/train_grpo2.jsonl --out {OUT} \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --video-dir /content/pt_videos --frames-dir /content/pt_frames \
    --group-size 8 --prompts-per-step 4 --micro-batch 4 --lr 2e-5 --beta 0.02 --numeric-partial 0.5 --max-draws 12 \
    --max-steps 20 --save-every 10

In [ ]:
# 学習の記録を表にする
import json, pandas as pd
rows = [json.loads(l) for l in open(f'{OUT}/train_log.jsonl')]
df = pd.DataFrame(rows)
print(df[['step','reward','acc','parsable','kl','loss','len','draws','zero_adv_groups','elapsed']].tail(20).to_string(index=False))
print('1 ステップあたり', round(df.elapsed.diff().median()), '秒')
w = df.groupby((df.step - 1) // 50).agg(reward=('reward','mean'), acc=('acc','mean'), kl=('kl','mean'), draws=('draws','mean'), skip=('zero_adv_groups','mean'))
print(w.round(3).to_string())

## 5. 学習後の dev（最新のチェックポイント）

In [ ]:
%cd /content/movie_llm
latest = open(f'{OUT}/latest.txt').read().strip()
print('adapter:', latest)
!python scripts/pt_difficulty.py --items data/perception_test/dev_grpo2.jsonl --out {OUT}/dev_after.jsonl \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --greedy --video-dir /content/pt_videos --frames-dir /content/pt_frames --adapter {latest}
!python scripts/pt_difficulty.py --items data/perception_test/dev_grpo2.jsonl --out {OUT}/dev_before.jsonl --summarize
!python scripts/pt_difficulty.py --items data/perception_test/dev_grpo2.jsonl --out {OUT}/dev_after.jsonl --summarize

## 6. ベンチマーク（4択・yes_no、32 フレーム）を学習後の LoRA で予測（約 15 分）

In [ ]:
%cd /content/movie_llm
latest = open(f'{OUT}/latest.txt').read().strip()
for t in ['multi_choice', 'yes_no']:
    !python scripts/jvqa_predict.py --name grpo2_4b_f32 --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --adapter {latest} \
        --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results --type {t}

終わったら、手順 4 の表と手順 5 の 2 つの集計を手元の私に貼る。持ち帰るもの:
- `MyDrive/movie_llm/grpo2_4b/` の `latest.txt` が指す LoRA フォルダと `train_log.jsonl` → 手元の `results/grpo2_4b/`
- `MyDrive/movie_llm/results/grpo2_4b_f32/predictions.jsonl` → 手元の `results/grpo2_4b_f32/`（`scripts/jvqa_eval.py --name grpo2_4b_f32` で採点）